# MobileNet Architecture Guide

**MobileNet** is a Convolutional Neural Network (CNN) architecture designed for efficient inference on resource-constrained devices, such as smartphones, embedded systems, and edge devices.

Its key idea is to replace expensive standard convolutions with **depthwise separable convolutions**.

### Main Goals

* Low inference latency
* Low computational cost
* Low memory usage
* Low power consumption
* Good prediction accuracy

---

## 1. Why Do We Need MobileNet?

A standard CNN can achieve high accuracy, but convolution operations can be computationally expensive. This becomes a problem when deploying models on:

* Smartphones, IoT devices, and embedded systems
* Edge devices and devices with limited CPU/GPU resources
* Battery-powered devices

While a data-center GPU can perform billions of operations efficiently, mobile devices have strict constraints on compute, memory, battery, heat, and latency.

Therefore, MobileNet asks: *Can we keep most of the useful feature-learning ability of a CNN while dramatically reducing computation?* The answer is to replace the expensive standard convolution with a depthwise separable convolution.

---

## 2. Standard Convolution

Suppose an input feature map has:

* Height = $H$
* Width = $W$
* Input channels = $M$

And we want $N$ output channels with a kernel size of $K \times K$.

A standard convolution performs two things at the same time: **spatial filtering** and **channel mixing**. For every output channel, the convolution looks at a $K \times K$ spatial region across all $M$ input channels.

### Mathematical Cost

* **Filter Dimensions:** $K \times K \times M$ per output filter.
* **Number of Parameters:** $K^2 M N$
* **Multiply-Accumulate Operations (MACs):** $HWK^2MN$

The expensive part is the combination $K^2MN$, because every output channel interacts with every input channel.

---

## 3. The Key Idea of MobileNet

Instead of performing spatial filtering and channel mixing in one expensive operation, MobileNet separates them into two cheaper steps:

$$\text{Depthwise Convolution} \rightarrow \text{Pointwise Convolution} = \text{Depthwise Separable Convolution}$$

* **Depthwise convolution** = Spatial filtering
* **Pointwise convolution** = Channel mixing

---

## 4. Depthwise Convolution

A depthwise convolution applies a spatial filter independently to each input channel. Instead of using one filter that looks at all $M$ channels simultaneously, it gives each channel its own spatial filter ($3 \times 3$ filter per channel).

* **Parameters:** $K^2M$ (Notice that the $N$ factor from standard convolution has disappeared).

### Intuition

Depthwise convolution asks: *"What spatial patterns exist within each feature channel?"* It learns edges, corners, textures, and local patterns. However, it **does not mix information between channels**, which requires a second operation.

---

## 5. Pointwise Convolution

After depthwise convolution, MobileNet uses a $1 \times 1$ convolution, known as a **pointwise convolution**.

A $1 \times 1$ convolution looks at all channels at a single spatial location to perform **channel mixing**:

$$y = w_1 x_1 + w_2 x_2 + \dots + w_M x_M + b$$

* **Parameters:** $MN$
* **Capabilities:** Can increase/decrease channel dimensions, mix features, and create new feature combinations.

---

## 6. Depthwise + Pointwise Comparison

| Property | Standard Convolution | Depthwise Separable Convolution |
| --- | --- | --- |
| **Spatial Filtering** | Yes | Yes |
| **Channel Mixing** | Yes | Yes |
| **Operation Style** | Done together (All-in-one) | Separated (Two-stage) |
| **Parameters** | $K^2MN$ | $K^2M + MN$ |
| **Computation** | $HWK^2MN$ | $HW(K^2M + MN)$ |
| **Efficiency** | More expensive | Much cheaper |

---

## 7. Why Is It So Much Cheaper? (Example)

Suppose:

* $K = 3$
* $M = 64$
* $N = 128$
* **Standard Convolution Parameters:** $3^2 \times 64 \times 128 = 73,728$
* **Depthwise Convolution Parameters:** $3^2 \times 64 = 576$
* **Pointwise Convolution Parameters:** $64 \times 128 = 8,192$
* **Total Depthwise Separable Parameters:** $576 + 8,192 = 8,768$

$$\text{Ratio} = \frac{73,728}{8,768} \approx 8.4\times \text{ fewer parameters!}$$

### General Computational Reduction

The computational ratio simplifies to:

$$\frac{K^2M + MN}{K^2MN} = \frac{1}{N} + \frac{1}{K^2}$$

For a typical $3 \times 3$ kernel, if $N$ is large, this approximates to $\frac{1}{9}$, meaning depthwise separable convolution reduces computation by **8 to 9 times**.

---

## 8. Key Activation & Normalization Layers

### ReLU (Rectified Linear Unit)

* **Definition:** $\text{ReLU}(x) = \max(0, x)$
* **Why we need it:** Without non-linear activation functions, stacking multiple linear layers collapses into a single linear transformation. ReLU introduces non-linearity, allowing the network to learn complex relationships and helping mitigate the vanishing gradient problem for positive inputs.

### Batch Normalization (BatchNorm)

Stabilizes training by normalizing activations using mini-batch statistics and applying learned scale and shift parameters. Benefits include faster convergence and more stable optimization.

---

## 9. Evolution: MobileNetV1 vs. MobileNetV2

### MobileNetV1

* **Core Idea:** Standard Conv $\rightarrow$ Depthwise Conv + Pointwise Conv.
* **Focus:** Efficient image classification via depthwise separable convolutions.

### MobileNetV2

Introduces two architectural improvements:

1. **Inverted Residual Blocks:** Expands channels first ($1 \times 1$), performs cheap depthwise processing, then projects back down to a low dimension.
2. **Linear Bottlenecks:** Removes non-linear activations (ReLU) right before the projection layer to prevent information loss in low-dimensional spaces.

| Feature | MobileNetV1 | MobileNetV2 |
| --- | --- | --- |
| **Depthwise Separable Conv** | Yes | Yes |
| **$1 \times 1$ Convolution** | Yes | Yes |
| **Inverted Residuals** | No | Yes |
| **Linear Bottlenecks** | No | Yes |

---

## 10. Summary Mental Model

* **Depthwise Conv:** Handles **"Where?"** (Spatial filtering per channel)
* **Pointwise Conv:** Handles **"What combination?"** (Channel mixing across features)
* **MobileNetV1:** Depthwise Separable Convolutions
* **MobileNetV2:** Inverted Residuals + Linear Bottlenecks + Expansion



In [ ]:
# Inference; using Hugging Face Transformers

from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import requests


url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image=Image.open(requests.get(url,stream=True).raw)


#initialize processor and model
preprocessor=AutoImageProcessor.from_pretrained("google/mobilenet_v2_1.0_224")
model=AutoModelForImageClassification.from_pretrained("google/mobilenet_v2_1.0_224")


# preprocess the inputs
inputs=preprocessor(images=image, return_tensors="pt")


# get the output and class labels
output=model(**inputs)
logits=output.logits

predicted_class_idx=logits.argmax(-1).item()
print("predicted class", model.config.id2label[predicted_class_idx])

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

predicted class tabby, tabby cat


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_channels,
        )
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, stride=1, padding=0
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


class MobileNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1)

        # MobileNet body
        self.dw_conv2 = DepthwiseSeparableConv(32, 64, 1)
        self.dw_conv3 = DepthwiseSeparableConv(64, 128, 2)
        self.dw_conv4 = DepthwiseSeparableConv(128, 128, 1)
        self.dw_conv5 = DepthwiseSeparableConv(128, 256, 2)
        self.dw_conv6 = DepthwiseSeparableConv(256, 256, 1)
        self.dw_conv7 = DepthwiseSeparableConv(256, 512, 2)

        # 5 depthwise separable convolutions with stride 1
        self.dw_conv8 = DepthwiseSeparableConv(512, 512, 1)
        self.dw_conv9 = DepthwiseSeparableConv(512, 512, 1)
        self.dw_conv10 = DepthwiseSeparableConv(512, 512, 1)
        self.dw_conv11 = DepthwiseSeparableConv(512, 512, 1)
        self.dw_conv12 = DepthwiseSeparableConv(512, 512, 1)

        self.dw_conv13 = DepthwiseSeparableConv(512, 1024, 2)
        self.dw_conv14 = DepthwiseSeparableConv(1024, 1024, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.dw_conv2(x)
        x = F.relu(x)
        x = self.dw_conv3(x)
        x = F.relu(x)
        x = self.dw_conv4(x)
        x = F.relu(x)
        x = self.dw_conv5(x)
        x = F.relu(x)
        x = self.dw_conv6(x)
        x = F.relu(x)
        x = self.dw_conv7(x)
        x = F.relu(x)

        x = self.dw_conv8(x)
        x = F.relu(x)
        x = self.dw_conv9(x)
        x = F.relu(x)
        x = self.dw_conv10(x)
        x = F.relu(x)
        x = self.dw_conv11(x)
        x = F.relu(x)
        x = self.dw_conv12(x)
        x = F.relu(x)

        x = self.dw_conv13(x)
        x = F.relu(x)
        x = self.dw_conv14(x)
        x = F.relu(x)

        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x


# Create the model
mobilenet = MobileNet(num_classes=1000)
print(mobilenet)

MobileNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (dw_conv2): DepthwiseSeparableConv(
    (depthwise): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32)
    (pointwise): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
  )
  (dw_conv3): DepthwiseSeparableConv(
    (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=64)
    (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
  )
  (dw_conv4): DepthwiseSeparableConv(
    (depthwise): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=128)
    (pointwise): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1))
  )
  (dw_conv5): DepthwiseSeparableConv(
    (depthwise): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=128)
    (pointwise): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1))
  )
  (dw_conv6): DepthwiseSeparableConv(
    (depthwise): Conv2d(256, 256, kernel_size=(3, 3)